# 1. Two Sum

[Problem](https://leetcode.com/problems/two-sum/) · difficulty: easy

Four accepted approaches, all correct, spread across a 95x runtime range. This notebook is about *why* the spread is that wide — two of the four are quadratic and still land far apart.


In [ ]:
import pathlib, sys; sys.path.insert(0, '../..')

from lc.harness import as_cases, load_solutions, run

solutions = load_solutions('.')
[s.__name__ for s in solutions]


## Summary

| Approach | Time | Space | Idea |
|---|---|---|---|
| `SolutionNestedLoops` | O(n²) | O(1) | every pair |
| `SolutionNestedLoopsSkipRepeats` | O(n²) | O(n) | every pair, each distinct left value used once |
| `SolutionComplementScan` | O(n²) | O(1) | inner loop delegated to `list.index` (C) |
| `SolutionComplementHashMap` | O(n) | O(n) | one pass, map of the complement each element waits for |


## The hash map pass, step by step

The map holds `target - nums[j] -> j` for every index `j` already visited. So `nums[i] in seen` reads as *"an earlier element is waiting for exactly this value"*, and the pair is `[seen[nums[i]], i]`.

Storing **after** the look-up is what forbids an element pairing with itself — the trace below shows the write always landing one step behind the check.


In [ ]:
def trace(nums, target):
    seen = {}
    for i, value in enumerate(nums):
        hit = seen.get(value)
        print(f'i={i} value={value:>3}  seen={seen}  waiting for {value}? {hit}')
        if hit is not None:
            print(f'  -> [{hit}, {i}]')
            return [hit, i]
        seen[target - value] = i

trace([3, 2, 4], 6)


### Why `[3, 3]` is the case that matters

With the write before the check, index 0 would find its own entry and answer `[0, 0]`. With the write after, the entry found at index 1 can only have been written at index 0.


In [ ]:
trace([3, 3], 6)


## Two quadratic solutions, 5x apart

`SolutionComplementScan` does not change the complexity — `list.index` is a linear scan, so the search is still O(n²). What changes is where the loop *runs*: inside CPython's C implementation instead of in interpreted bytecode.

The measurement below uses the problem's own cases, so it can never drift from what the tests verify.


In [ ]:
from lc.bench import run_problem

for name, elapsed in sorted(run_problem(pathlib.Path('.').resolve(), repeat=200), key=lambda r: r[1]):
    print(f'{name:<32} {elapsed:7.3f} s')


### The same picture as input grows

The worst case for every approach is the answer sitting at the very end. Doubling `n` should roughly quadruple the three quadratic approaches and merely double the hash map.


In [ ]:
import time

def timed(solution, n):
    case = as_cases([((list(range(n)), 2 * n - 3), [n - 2, n - 1])])[0]
    start = time.perf_counter()
    run(solution, case)
    return (time.perf_counter() - start) * 1e3

sizes = [500, 1000, 2000]
print('worst case (answer at the end), milliseconds')
print(f"{'approach':<32}" + ''.join(f'{f"n={n}":>10}' for n in sizes))
for solution in solutions:
    row = ''.join(f'{timed(solution, n):10.2f}' for n in sizes)
    print(f'{solution.__name__:<32}{row}')


## Takeaway

- Complexity sets the slope; the constant factor sets where the line starts. `SolutionComplementScan` beats the interpreted double loop by ~5x on LeetCode and still loses to the hash map by two orders of magnitude, because the slope wins as soon as `n` is large enough.
- Skipping repeated left values (`SolutionNestedLoopsSkipRepeats`) is a correctness argument, not just an optimization: a later duplicate's partner is always to the right of the first occurrence too, so nothing is lost.
- `except ValueError`, never a bare `except` — the bare version also swallows the bugs you want to see.
